# Day 49 — Customer Intelligence Platform Baseline System
Integrates preprocessing, feature engineering, ML training, evaluation, and prediction outputs.

In [1]:
from pathlib import Path
import sys, json, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))
from src.preprocessing import engineer_features, build_preprocessor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
df = pd.read_csv(ROOT/'data/raw/customer_data.csv')
df = engineer_features(df)
df.head()

,customer_id,age,income,tenure_months,orders,total_spend,preferred_channel,city,complaints,churn,avg_order_value,orders_per_month,spend_per_month
0,C001,22.0,35000.0,6,4,12000,Online,Chennai,1,1,3000.000000,0.666667,2000.000000
1,C002,31.0,52000.0,18,12,45000,Mobile,Bengaluru,0,0,3750.000000,0.666667,2500.000000
2,C003,NaN,68000.0,24,18,72000,Online,Chennai,0,0,4000.000000,0.750000,3000.000000
3,C004,45.0,42000.0,30,9,31000,Store,Hyderabad,2,0,3444.444444,0.300000,1033.333333
4,C005,29.0,NaN,12,7,22000,Mobile,Bengaluru,1,1,3142.857143,0.583333,1833.333333


## Baseline objective
Predict customer churn using a simple Logistic Regression baseline. The model is intentionally interpretable and establishes a measurable benchmark before more advanced models are introduced.

In [3]:
X = df.drop(columns=['churn','customer_id'])
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
preprocessor, numeric, categorical = build_preprocessor(X_train)
model = Pipeline([('preprocessor',preprocessor),('classifier',LogisticRegression(max_iter=1000,class_weight='balanced'))])
model.fit(X_train,y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]
metrics = {'accuracy':accuracy_score(y_test,pred),'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0),'f1':f1_score(y_test,pred,zero_division=0),'roc_auc':roc_auc_score(y_test,prob)}
metrics

{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'roc_auc': 1.0}

In [4]:
results = X_test.copy()
results['customer_id'] = df.loc[X_test.index,'customer_id']
results['actual_churn'] = y_test
results['predicted_churn'] = pred
results['churn_probability'] = prob
results[['customer_id','actual_churn','predicted_churn','churn_probability']].sort_values('churn_probability',ascending=False)

,customer_id,actual_churn,predicted_churn,churn_probability
45,C046,1,1,0.989961
38,C039,1,1,0.983530
35,C036,1,1,0.968767
24,C025,1,1,0.939186
40,C041,1,1,0.795226
31,C032,0,0,0.240423
11,C012,0,0,0.020806
49,C050,0,0,0.019348
12,C013,0,0,0.000911
23,C024,0,0,0.000056


## Interpretation
This baseline is not the final model. Its purpose is to validate the end-to-end capstone flow and establish a benchmark. Future iterations can compare tree-based models, calibration, threshold tuning, segmentation, and time-aware validation.